In [ ]:
import sys
import backtrader as bt
import logging
import numpy as np
import json
import os
import pandas as pd
import yfinance as yf
from statsmodels.tsa.stattools import coint
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from plotly.subplots import make_subplots
from datetime import datetime
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.signal import argrelextrema
import plotly.express as px
import plotly.graph_objects as go
import warnings
import vectorbt as vbt
import yfinance as yf
# Custom library paths and imports
sys.path.append(r"e:\Coding Projects\Investment Analysis")
from Quantapp.visualization import Plotter
from Quantapp.analytics import TimeSeriesAnalytics as Rolling, Helper, Models
from Quantapp.data import MacroDataClient

# Disable yfinance logging and warnings
logger = logging.getLogger('yfinance')
logger.disabled = True
logger.propagate = False
warnings.filterwarnings("ignore")

# Instantiate Quantapp objects
qc = Rolling()
qp = Plotter()
qe = MacroDataClient()
helper = Helper()
model = Models()


In [ ]:
#PARAMETERS (Adjust parmeters here)
# Define parameters for the analysis (Adjust these as needed)
interval = '1d'
period     = '20y'
risk_free_rate = 0.02 / 252  # Annualized risk-free rate divided by trading days
#should be a string or None
ticker_str = 'SPY' #ticker to analyze
benchmark_str = 'SPY'# #primary benchmark to compare against
init_cash = 100000  # Initial cash for backtesting


In [ ]:
#Load data
yf_ticker = yf.Ticker(ticker_str)
price = vbt.YFData.download(ticker_str, period=period, interval=interval)
price_close = price.get('Close')
#price_close = yf_ticker.history(period=period, interval=interval)['Close']
#price_close.head()

In [ ]:
#--- Backtest a buy-and-hold strategy ---
fig = go.Figure()
import vectorbt as vbt
import yfinance as yf

#--- Backtest a buy-and-hold strategy ---
entries = price_close.index == price_close.index[0]  # Only first date is True
exits = [False] * len(price_close)  # Never exit until end



# Run the portfolio simulation
portfolio = vbt.Portfolio.from_signals(
    close=price_close,
    entries=entries,
    exits=exits,
    init_cash=init_cash,
    size_type='percent',
    size=1.0,  # Buy full cash
)

# Show performance
#print(portfolio.stats())
portfolio.plot().show()



In [ ]:
#--- Backtest a simple moving average crossover strategy ---
short_sma = vbt.MA.run(price_close, window=50).ma
long_sma = vbt.MA.run(price_close, window=200).ma

# Generate entry and exit signals
entries = short_sma > long_sma
exits = short_sma < long_sma

#sizing: buy/sell as many shares as possible when a signal is triggered
shares_to_buy = init_cash / price_close.iloc[0]

# Run portfolio simulation
portfolio = vbt.Portfolio.from_signals(
    close=price_close,
    entries=entries,
    exits=exits,
    init_cash=10000,
    size_type='percent',
    size=1.0
)

# Show performance
#print(portfolio.stats())
portfolio.plot().show()

In [ ]:
#--- Backtest a RSI strategy ---
# --- Compute 14-day RSI ---
rsi = vbt.RSI.run(price_close, window=14).rsi

# --- Generate entry and exit signals ---
# Buy when RSI < 30 (oversold)
entries = rsi < 30

# Sell when RSI > 70 (overbought)
exits = rsi > 70

# --- Portfolio simulation ---
portfolio = vbt.Portfolio.from_signals(
    close=price_close,
    entries=entries,
    exits=exits,
    init_cash=init_cash,
    size_type='percent',
    size=1.0  # deploy 100% cash when signal triggers
)

# Show performance
print(portfolio.stats())
portfolio.plot().show()


In [ ]:
#--- Backtest a perfect timing strategy (crystal ball) ---

# --- Find local minima (buy points) and maxima (sell points) ---
window = 10  # lookback/lookahead window

local_min = argrelextrema(price_close.values, np.less, order=window)[0]
local_max = argrelextrema(price_close.values, np.greater, order=window)[0]

# Generate signals
entries = np.zeros(len(price_close), dtype=bool)
exits = np.zeros(len(price_close), dtype=bool)

entries[local_min] = True
exits[local_max] = True

# --- Portfolio simulation ---
portfolio = vbt.Portfolio.from_signals(
    close=price_close,
    entries=entries,
    exits=exits,
    init_cash=init_cash,
    size_type='percent',
    size=1.0
)

# Show performance
print(portfolio.stats())
portfolio.plot().show()


In [ ]:
import vectorbt as vbt
import yfinance as yf

# --------------------------
# Parameters
# --------------------------
ticker = "SPY"
period = "10y"         # backtest period
lookback = 21          # n-day returns (21 trading days ~ 1 month)
window = 21            # rolling window for mean/std
z_entry = -3           # buy threshold
z_exit =  3            # sell threshold
init_cash = 10_000

# --------------------------
# Load data
# --------------------------
price = yf.Ticker(ticker).history(period=period)['Close']

# --------------------------
# Compute returns & z-scores
# --------------------------
n_day_returns = price.pct_change(lookback)

rolling_mean = n_day_returns.rolling(window).mean()
rolling_std = n_day_returns.rolling(window).std()

zscore = (n_day_returns - rolling_mean) / rolling_std

# --------------------------
# Trading signals
# --------------------------
entries = zscore < z_entry     # Buy when extremely negative
exits   = zscore > z_exit      # Sell/exit when extremely positive

# --------------------------
# Backtest
# --------------------------
portfolio = vbt.Portfolio.from_signals(
    close=price,
    entries=entries,
    exits=exits,
    init_cash=init_cash,
    size_type='percent',   # allocate full capital
    size=1.0
)

# --------------------------
# Results
# --------------------------
print(portfolio.stats())
portfolio.plot().show()
